# Bronze Layer Ingestion
```
BRONZE LAYER
  customers, products, orders, inventory  → CSV via Auto Loader
  clickstream                             → JSON via Auto Loader
          ↓
SILVER LAYER  (clean individual tables)
  silver.customers
  silver.products
  silver.orders
  silver.inventory
  silver.clickstream
          ↓
SILVER MART  (multi-table joins)
  silver.orders_clickstream
  silver.inventory_orders
  silver.customer_360
          ↓
GOLD LAYER  (aggregations + KPIs)
  gold.revenue_by_category
  gold.funnel_analysis
  gold.inventory_health
  gold.customer_segments
  gold.daily_metrics
```

## Cell 1 — Create Catalog & Schemas

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce;
USE CATALOG ecommerce;

CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

## Cell 2 — Imports & Shared CSV Options

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, to_timestamp, lit

# ── Shared options for CSV tables (customers / products / orders / inventory) ──
base_options = {
    "cloudFiles.format"        : "csv",
    "cloudFiles.schemaLocation": "s3://ecommerce-lakehouse-databricks/schemas/csv/",
    "header"                   : "true",
    "inferSchema"              : "true",
}

print("✅ Imports and base_options ready")

✅ Imports and base_options ready


## Cell 3 — Bronze: Customers (CSV)

In [0]:
%sql
-- Clear out the target tables to ensure a clean, fresh load
TRUNCATE TABLE bronze.customers;
TRUNCATE TABLE bronze.products;
TRUNCATE TABLE bronze.orders;
TRUNCATE TABLE bronze.inventory;

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType, DoubleType, IntegerType
from pyspark.sql.functions import col, current_timestamp

# ══════════════════════════════════════════════════════════════════════════════
# DATA SCHEMAS DEFINITION
# ══════════════════════════════════════════════════════════════════════════════

# --- CUSTOMERS ---
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", StringType(), True),  
    StructField("is_active", BooleanType(), True),
    # Auto Loader Metadata Fields
    StructField("ingestion_time", TimestampType(), True),
    StructField("source_file", StringType(), True)
])

# --- PRODUCTS ---
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("cost_price", DoubleType(), True),
    StructField("stock_qty", IntegerType(), True),
    # Auto Loader Metadata Fields
    StructField("ingestion_time", TimestampType(), True),
    StructField("source_file", StringType(), True)
])

# --- ORDERS ---
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", StringType(), True),  
    StructField("payment_method", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", DoubleType(), True),
    # Auto Loader Metadata Fields
    StructField("ingestion_time", TimestampType(), True),
    StructField("source_file", StringType(), True)
])

# --- INVENTORY ---
inventory_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("warehouse_id", StringType(), True),
    StructField("available_qty", IntegerType(), True),
    StructField("last_updated", StringType(), True), 
    # Auto Loader Metadata Fields
    StructField("ingestion_time", TimestampType(), True),
    StructField("source_file", StringType(), True)
])


# ══════════════════════════════════════════════════════════════════════════════
# STREAMING PIPELINES (BRONZE INGESTION)
# ══════════════════════════════════════════════════════════════════════════════

# --- CUSTOMERS STREAM ---
customers_df = (
    spark.readStream
    .format("cloudFiles")
    .options(**base_options)
    .schema(customers_schema)
    .option("pathGlobFilter", "customers.csv")
    .option("cloudFiles.schemaLocation", "s3://ecommerce-lakehouse-databricks/schemas/customers/")
    .load("s3://ecommerce-lakehouse-databricks/batch/")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

customers_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "s3://ecommerce-lakehouse-databricks/checkpoints/customers_v5/") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("bronze.customers")


# --- PRODUCTS STREAM ---
products_df = (
    spark.readStream
    .format("cloudFiles")
    .options(**base_options)
    .schema(products_schema)
    .option("pathGlobFilter", "products.csv")
    .option("cloudFiles.schemaLocation", "s3://ecommerce-lakehouse-databricks/schemas/products/")
    .load("s3://ecommerce-lakehouse-databricks/batch/")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

products_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "s3://ecommerce-lakehouse-databricks/checkpoints/products_v5/") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("bronze.products")


# --- ORDERS STREAM ---
orders_df = (
    spark.readStream
    .format("cloudFiles")
    .options(**base_options)
    .schema(orders_schema)
    .option("pathGlobFilter", "orders.csv")
    .option("cloudFiles.schemaLocation", "s3://ecommerce-lakehouse-databricks/schemas/orders/")
    .load("s3://ecommerce-lakehouse-databricks/batch/")
    .withColumn("withColumn", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

orders_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "s3://ecommerce-lakehouse-databricks/checkpoints/orders_v5/") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("bronze.orders")


# --- INVENTORY STREAM ---
inventory_df = (
    spark.readStream
    .format("cloudFiles")
    .options(**base_options)
    .schema(inventory_schema)
    .option("pathGlobFilter", "inventory.csv")
    .option("cloudFiles.schemaLocation", "s3://ecommerce-lakehouse-databricks/schemas/inventory/")
    .load("s3://ecommerce-lakehouse-databricks/batch/")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

inventory_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", "s3://ecommerce-lakehouse-databricks/checkpoints/inventory_v5/") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("bronze.inventory")

In [0]:
tables = ["customers", "products", "orders", "inventory","clickstream"]

for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: bronze.{table}")
    print(f"{'='*50}")
    
    df = spark.table(f"bronze.{table}")
    
    print("\nColumns:")
    print(df.columns)
    
    print("\nSchema:")
    df.printSchema()


TABLE: bronze.customers

Columns:


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8903113526394753>, line 11
      8 df = spark.table(f"bronze.{table}")
     10 print("\nColumns:")
---> 11 print(df.columns)
     13 print("\nSchema:")
     14 df.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:309, in DataFrame.columns(self)
    307 @property
    308 def columns(self) -> List[str]:
--> 309     return [field.name for field in self._schema.fields]

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1977, in DataFrame._schema(self)
   1975 if self._cached_schema is None:
   1976     query = self._plan.to_proto(self._session.client)
-> 1977     self._cached_schema = self._session.client.schema(query)
   1978     try:
   1979         self._cached_schema_serialized = CPickleSerializer().dumps(self._schema)

File /databricks/

In [0]:
%sql
SHOW TABLES IN ecommerce.bronze;

database,tableName,isTemporary
bronze,clickstream,false
bronze,customers,false
bronze,inventory,false
bronze,orders,false
bronze,products,false


## Cell 7 — Bronze: Clickstream (JSON from S3)
Files were uploaded by `upload_to_s3.py` to `s3://ecommerce-lakehouse-databricks/stream/`

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

from pyspark.sql.functions import (
    to_timestamp,
    current_timestamp,
    col,
    lit
)

# ── Paths ─────────────────────────────────────────────
CLICKSTREAM_SOURCE     = "s3://ecommerce-lakehouse-databricks/stream/"
CLICKSTREAM_CHECKPOINT = "s3://ecommerce-lakehouse-databricks/checkpoints/clickstream/"
CLICKSTREAM_TABLE      = "ecommerce.bronze.clickstream"

# ── Schema ────────────────────────────────────────────
clickstream_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("page_url", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("referrer", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),
    StructField("amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("time_spent_sec", IntegerType(), True),
    StructField("timestamp", StringType(), True)
])

clickstream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        CLICKSTREAM_CHECKPOINT + "schema/"
    )
    .option("multiLine", "true")
    .schema(clickstream_schema)
    .load(CLICKSTREAM_SOURCE)

    .withColumn(
        "event_ts",
        to_timestamp(
            "timestamp",
            "yyyy-MM-dd'T'HH:mm:ss"
        )
    )

    .withColumn(
        "ingested_at",
        current_timestamp()
    )

    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )

    .withColumn(
        "source_system",
        lit("flask_clickstream_api")
    )
)

query = (
    clickstream_df.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        CLICKSTREAM_CHECKPOINT
    )
    .outputMode("append")

    # continuously check for files every 30 sec
    #.trigger(processingTime="30 seconds")
    .trigger(availableNow=True)
    .table(CLICKSTREAM_TABLE)
)

print("Streaming started...")
print(query.id)

In [0]:
query.status

{'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

In [0]:
query.isActive

False

In [0]:
query.lastProgress

{
    "id": "ef7b0cc8-3237-40c9-b947-9be0afb71d07",
    "runId": "0c4af18f-26a0-41ca-be33-5587f139fc4c",
    "name": null,
    "timestamp": "2026-06-02T09:24:58.862Z",
    "batchId": 0,
    "batchDuration": 5892,
    "durationMs": {
        "triggerExecution": 5892,
        "queryPlanning": 356,
        "collectSourceMetrics": 0,
        "getBatch": 134,
        "commitOffsets": 69,
        "postCommitBatchCallback": 1,
        "addBatch": 3615,
        "latestOffset": 1381,
        "postAddBatchCallback": 1,
        "commitBatch": 247,
        "walCommit": 120
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "CloudFilesSource[s3://ecommerce-lakehouse-databricks/stream/]",
            "startOffset": null,
            "endOffset": "{\"seqNum\":4,\"sourceVersion\":3,\"lastBackfillStartTimeMs\":1780392298925,\"lastBackfillFinishTimeMs\":1780392300240,\"lastInputPath\":\"s3://ecommerce-lakehouse-databricks/stream/\"}",
            

In [0]:
spark.sql("""
SELECT
COUNT(DISTINCT source_file) files_loaded
FROM ecommerce.bronze.clickstream
""").show()

+------------+
|files_loaded|
+------------+
|           2|
+------------+



In [0]:
display(
spark.sql("""
SELECT source_file,
COUNT(*) rows_loaded
FROM ecommerce.bronze.clickstream
GROUP BY source_file
""")
)

source_file,rows_loaded
s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,150000
s3://ecommerce-lakehouse-databricks/stream/file_1_20260602_104922.json,150000


In [0]:
display(
    spark.table("ecommerce.bronze.clickstream")
    .limit(10)
)

event_id,event_type,user_id,session_id,product_id,order_id,page_url,device_type,referrer,quantity,unit_price,discount_pct,amount,payment_method,time_spent_sec,timestamp,event_ts,ingested_at,source_file,source_system
E00150188,page_view,C028942,S11116D22,null,null,/cart,tablet,google,null,null,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150189,add_to_cart,C020567,S20258AD0,P001710,null,null,desktop,null,2,4089.53,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150190,page_view,C029421,SBF4E2AD4,null,null,/cart,tablet,email_campaign,null,null,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150191,product_view,C008952,S24A61ECC,P001557,null,/product/P001557,mobile,null,null,null,null,null,null,70,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150192,product_view,C013459,S5B9962E8,P006598,null,/product/P006598,desktop,null,null,null,null,null,null,5,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150193,page_view,C049637,S4EC358D6,null,null,/category/electronics,desktop,instagram,null,null,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150194,add_to_cart,C003454,S23C80689,P002769,null,null,mobile,null,4,4221.44,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150195,product_view,C014384,SE1158048,P002491,null,/product/P002491,mobile,null,null,null,null,null,null,117,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150196,page_view,C040955,S10E61F76,null,null,/deals,desktop,instagram,null,null,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api
E00150197,add_to_cart,C029153,S38BCF008,P007845,null,null,tablet,null,3,1497.24,null,null,null,null,2026-06-02T05:19:46,2026-06-02T05:19:46.000Z,2026-06-02T09:25:00.244Z,s3://ecommerce-lakehouse-databricks/stream/file_2_20260602_105017.json,flask_clickstream_api


In [0]:
tables = spark.sql(
    "SHOW TABLES IN ecommerce.bronze"
).collect()

for t in tables:
    
    table_name = t.tableName
    
    print("\n" + "="*70)
    print(f"TABLE: ecommerce.bronze.{table_name}")
    print("="*70)
    
    df = spark.table(
        f"ecommerce.bronze.{table_name}"
    )
    
    for field in df.schema.fields:
        print(
            f"Column: {field.name:<25} "
            f"Datatype: {field.dataType} "
            f"Nullable: {field.nullable}"
        )


TABLE: ecommerce.bronze.clickstream
Column: event_id                  Datatype: StringType() Nullable: True
Column: event_type                Datatype: StringType() Nullable: True
Column: user_id                   Datatype: StringType() Nullable: True
Column: session_id                Datatype: StringType() Nullable: True
Column: product_id                Datatype: StringType() Nullable: True
Column: order_id                  Datatype: StringType() Nullable: True
Column: page_url                  Datatype: StringType() Nullable: True
Column: device_type               Datatype: StringType() Nullable: True
Column: referrer                  Datatype: StringType() Nullable: True
Column: quantity                  Datatype: IntegerType() Nullable: True
Column: unit_price                Datatype: DoubleType() Nullable: True
Column: discount_pct              Datatype: DoubleType() Nullable: True
Column: amount                    Datatype: DoubleType() Nullable: True
Column: payment_method    